In [15]:
from __future__ import annotations
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

In [2]:
df = pd.read_parquet("/Users/YGT/ist-airport-decision-support-system/data/gold/trajectories/trajectory_gold.parquet")

In [12]:
df.columns

Index(['flight_id', 'episode_id', 'point_timestamp', 'icao', 'date', 'e_m',
       'n_m', 'u_m', 'ux', 'uy', 'uz', 'r', 'sin_theta', 'cos_theta',
       'delta_t', 'distance_km', 'segment_id', 'gap_flag'],
      dtype='str')

In [16]:
df = df.sort_values(["flight_id", "point_timestamp", "icao"], kind="mergesort").reset_index(drop=True)

In [ ]:


FEATURE_COLS = [
    "e_m","n_m","u_m",
    "ux","uy","uz",
    "r","sin_theta","cos_theta",
    "delta_t",
]

REQ_COLS = ["flight_id", "point_timestamp", "segment_id"] + FEATURE_COLS


class ATSCCFlightDataset(Dataset):
    """
    Fast in-memory dataset for ATSCC.

    Assumptions (you already verified):
      - Parquet is globally sorted by ["flight_id", "point_timestamp"]
      - Each flight_id occurs as one contiguous block (no fragmentation)

    Each sample = one flight_id trajectory:
      x:   (T, 10) float32
      seg: (T,)    int64
    """
    def __init__(self, parquet_path: str, keep_meta: bool = True):
        self.parquet_path = parquet_path
        self.keep_meta = keep_meta

        df = pd.read_parquet(parquet_path, columns=REQ_COLS)

        flight = df["flight_id"].to_numpy()

        change = np.ones(len(df), dtype=bool)
        change[1:] = flight[1:] != flight[:-1]
        starts = np.flatnonzero(change)
        ends = np.r_[starts[1:], len(df)]
        self.offsets = np.stack([starts, ends], axis=1).astype(np.int64)

        self.X_all = df[FEATURE_COLS].to_numpy(np.float32, copy=True)
        self.seg_all = df["segment_id"].to_numpy(np.int64, copy=True)

        if keep_meta:
            self.flight_list = df.loc[starts, "flight_id"].tolist()

    def __len__(self) -> int:
        return int(self.offsets.shape[0])

    def __getitem__(self, idx: int):
        s, e = self.offsets[idx]
        x = self.X_all[s:e]       # (T,10)
        seg = self.seg_all[s:e]   # (T,)

        out = {
            "x": torch.from_numpy(x),      # float32
            "seg": torch.from_numpy(seg),  # int64
            "len": int(e - s),
        }
        if self.keep_meta:
            out["flight_id"] = self.flight_list[idx]
        return out

0    1
1    1
2    1
3    1
4    1
5    1
6    1
7    1
8    1
9    1
Name: segment_id, dtype: int32